# DecompDiff local sweep 3 — best config across datasets

Fixed **winning objective** for every run: **MSE loss, predict $x_0$, loss weight ON**,
architecture NL1 / NF1 / hidden\_dim 64 (same as the stock ablation).

**Part 1 — new datasets** (batch size 64): etth1, etth2, exchange, fmri, eeg, sine.
Channels are set automatically per dataset.

**Part 2 — stock batch-size sweep**: batch size 24, 64, 128.

Each cell is an independent wandb run (project `decompdiff-sweep3`). Run one at a time.

In [1]:
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from pathlib import Path
import wandb

REPO = Path("C:/Users/ameli/Desktop/TezBaselines")
sys.path.insert(0, str(REPO))

from DecompDiff.models.decompDiff import DecompDiff
from DecompDiff.models.diffusion  import GaussianDiffusion
from DecompDiff.config.stocks_config import Config
from DecompDiff.data.datasets import make_loaders
from MyCode.eval_metrics import evaluate_samples, vds_score, fdds_score, correlational_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

device: cuda


In [3]:
def compute_loss(model, diffusion, x_0, loss_type="mse",
                 prediction_type="x0", use_loss_weight=True):
    B = x_0.shape[0]
    t = torch.randint(0, diffusion.num_timesteps, (B,), device=x_0.device)
    x_t, noise = diffusion.q_sample(x_0, t)
    model_out = model(x_t, t)
    target = x_0 if prediction_type == "x0" else noise
    if loss_type == "l1":
        loss = F.l1_loss(model_out, target, reduction="none")
    else:
        loss = F.mse_loss(model_out, target, reduction="none")
    loss = loss.mean(dim=[1, 2])
    if use_loss_weight:
        loss = loss * diffusion.loss_weight[t]
    return loss.mean()


def compute_inline_metrics(model, diffusion, real_loader, device, num_steps=50,
                           prediction_type="x0"):
    model.eval()
    batches = [b.cpu().numpy() for b in real_loader]
    real_CL = np.concatenate(batches, axis=0)   # (N, C, L)
    N = real_CL.shape[0]
    chunk, chunks = 256, []
    with torch.no_grad():
        for start in range(0, N, chunk):
            bs = min(chunk, N - start)
            chunks.append(
                model.sample(diffusion, batch_size=bs, num_steps=num_steps, eta=0.0,
                             prediction_type=prediction_type).cpu()
            )
    fake_CL = torch.cat(chunks, dim=0).numpy()
    real_m = ((real_CL.transpose(0, 2, 1) + 1.0) * 0.5).astype(np.float32)
    fake_m = ((fake_CL.transpose(0, 2, 1) + 1.0) * 0.5).astype(np.float32)
    try:
        results = evaluate_samples(real_m, fake_m, device=device, n_iterations=3,
                                   disc_iterations=2000, pred_iterations=5000)
        vds  = vds_score(real_m, fake_m)
        fdds = fdds_score(real_m, fake_m)
        corr = correlational_score(real_m, fake_m)
    except Exception as e:
        print(f"  [metrics] failed: {e}"); model.train(); return None
    model.train()
    return {
        "disc_score":          results["discriminative"]["mean"],
        "disc_score_std":      results["discriminative"]["std"],
        "test_acc":            results["discriminative"]["test_acc"],
        "pred_mae":            results["predictive"]["mean"],
        "pred_mae_std":        results["predictive"]["std"],
        "vds": vds, "fdds": fdds, "correlational_score": corr,
    }

In [4]:
def run_experiment(dataset, window_length, num_epochs, num_layers=1,
                   num_fusion_layers=1, hidden_dim=64, batch_size=None,
                   eval_every=500, run_name=None):
    """Winning objective (MSE - predict x0 - loss weight ON) on any dataset.

    dataset    : stock, etth1, etth2, exchange, fmri, eeg, sine
    batch_size : override cfg default (used for the stock batch-size sweep).
    input_channels is set automatically from the dataset.
    """
    LOSS, PRED, WEIGHT = "mse", "x0", True          # fixed winning config

    cfg = Config()
    cfg.model.sequence_length = window_length
    cfg.training.num_epochs   = num_epochs
    cfg.data.dataset          = dataset
    cfg.model.num_layers        = num_layers
    cfg.model.num_fusion_layers = num_fusion_layers
    cfg.model.hidden_dim        = hidden_dim
    cfg.training.loss_type       = LOSS
    cfg.training.prediction_type = PRED
    cfg.training.use_loss_weight = WEIGHT
    if batch_size is not None:
        cfg.training.batch_size = batch_size
    bs = cfg.training.batch_size

    train_loader, _, ds = make_loaders(
        dataset, batch_size=bs, window=window_length,
        train_ratio=cfg.data.train_split, neg_one_to_one=cfg.data.neg_one_to_one,
        per_window=cfg.data.per_window_norm, num_workers=cfg.data.num_workers,
        pin_memory=False, data_root=cfg.data.data_root,
        sine_num=cfg.data.sine_num, sine_dim=cfg.data.sine_dim, seed=cfg.data.sine_seed,
    )
    cfg.model.input_channels = ds.num_features

    run_name = run_name or (
        f"decompdiff-{dataset}-L{window_length}-H{hidden_dim}"
        f"-NL{num_layers}-NF{num_fusion_layers}-mse-x0-w1-bs{bs}-E{num_epochs}"
    )

    wandb.init(
        project="decompdiff-sweep3", group="mse-x0-w1-datasets", name=run_name,
        tags=[dataset, f"bs{bs}", f"NL{num_layers}", f"NF{num_fusion_layers}",
              "loss-mse", "pred-x0", "w1"],
        config={**cfg.model.__dict__, **cfg.diffusion.__dict__, **cfg.training.__dict__,
                "dataset": dataset, "device": DEVICE, "eval_every": eval_every},
    )
    print(f"[{run_name}] batches={len(train_loader)} channels={cfg.model.input_channels} "
          f"batch_size={bs}")

    model = DecompDiff(
        input_channels=cfg.model.input_channels, sequence_length=window_length,
        hidden_dim=hidden_dim, num_heads=cfg.model.num_heads, num_layers=num_layers,
        num_fusion_layers=num_fusion_layers, mlp_ratio=cfg.model.mlp_ratio,
        dropout=cfg.model.dropout, freq_dim=cfg.model.freq_dim,
    ).to(DEVICE)
    diffusion = GaussianDiffusion(
        num_timesteps=cfg.diffusion.num_timesteps, beta_start=cfg.diffusion.beta_start,
        beta_end=cfg.diffusion.beta_end, noise_schedule=cfg.diffusion.noise_schedule,
        device=DEVICE,
    ).to(DEVICE)

    counts = model.get_parameter_count()
    print(f"[{run_name}] model_dim={model.model_dim} params={counts['total']:,}")
    wandb.config.update({"total_params": counts["total"], "model_dim": model.model_dim},
                        allow_val_change=True)

    total_steps = len(train_loader) * num_epochs
    optimizer = AdamW(model.parameters(), lr=cfg.training.learning_rate,
                      weight_decay=cfg.training.weight_decay, betas=(0.9, 0.999))
    warmup = LinearLR(optimizer, start_factor=1e-3, end_factor=1.0,
                      total_iters=cfg.training.warmup_steps)
    cosine = CosineAnnealingLR(optimizer, T_max=max(1, total_steps - cfg.training.warmup_steps),
                               eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine],
                             milestones=[cfg.training.warmup_steps])

    ckpt_dir = REPO / f"DecompDiff/output/checkpoints/{run_name}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    save_every = max(100, num_epochs // 5)

    for epoch in range(num_epochs):
        model.train(); epoch_loss = 0.0
        for batch in train_loader:
            x_0 = batch.to(DEVICE)
            optimizer.zero_grad()
            loss = compute_loss(model, diffusion, x_0, LOSS,
                                prediction_type=PRED, use_loss_weight=WEIGHT)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.training.gradient_clip_val)
            optimizer.step(); scheduler.step(); epoch_loss += loss.item()

        train_loss = epoch_loss / len(train_loader)
        current_lr = optimizer.param_groups[0]["lr"]
        log = {"train_loss": train_loss, "lr": current_lr, "epoch": epoch + 1}

        if (epoch + 1) % save_every == 0 or (epoch + 1) == num_epochs:
            torch.save({"epoch": epoch + 1, "train_loss": train_loss,
                        "model_state_dict": model.state_dict(),
                        "config": {"model": cfg.model.__dict__, "window_length": window_length,
                                   "dataset": dataset, "prediction_type": PRED,
                                   "use_loss_weight": WEIGHT, "loss_type": LOSS}},
                       ckpt_dir / f"checkpoint_ep{epoch+1}.pt")

        if (epoch + 1) % eval_every == 0:
            print(f"  [epoch {epoch+1}] metrics...")
            metrics = compute_inline_metrics(model, diffusion, train_loader, DEVICE,
                                             num_steps=50, prediction_type=PRED)
            if metrics is not None:
                log.update(metrics)
                print(f"  disc={metrics['disc_score']:.4f} pred={metrics['pred_mae']:.4f} "
                      f"vds={metrics['vds']:.4f} fdds={metrics['fdds']:.4f} "
                      f"corr={metrics['correlational_score']:.4f}")
        wandb.log(log)
        print(f"[{run_name}] ep {epoch+1:4d}/{num_epochs} train={train_loss:.5f} lr={current_lr:.2e}")

    print(f"[{run_name}] done. ckpt -> {ckpt_dir}")
    wandb.finish()
    return model, diffusion

In [5]:
# ── shared settings (winning config: MSE - predict x0 - loss weight ON) ───────
# architecture fixed at NL1 / NF1 / hidden_dim 64, same as the stock ablation.
WINDOW     = 32     # benchmarks (WaveletDiff / Diffusion-TS) use 24 -> set to 24
                    # if you want apples-to-apples with published numbers.
EPOCHS     = 2500
EVAL_EVERY = 500

## Part 1 — new datasets (batch size 64)

In [6]:
# ── etth1 · MSE · x0 · loss weight ON ──
run_experiment(dataset="etth1", window_length=WINDOW, num_epochs=EPOCHS,
               num_layers=1, num_fusion_layers=1, hidden_dim=64, batch_size=32,
               eval_every=EVAL_EVERY)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\ameli\_netrc.


StockDataset: 17389 windows  (train=17389, test=all [train_ratio=1.0])


wandb: Currently logged in as: a-meliksahdemir (a-meliksahdemir-bo-azi-i-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


[decompdiff-etth1-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] batches=544 channels=7 batch_size=32
[decompdiff-etth1-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] model_dim=64 params=252,807
[decompdiff-etth1-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    1/2500 train=0.14640 lr=1.00e-04
[decompdiff-etth1-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    2/2500 train=0.06007 lr=1.00e-04
[decompdiff-etth1-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    3/2500 train=0.05726 lr=1.00e-04
[decompdiff-etth1-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    4/2500 train=0.05590 lr=1.00e-04
[decompdiff-etth1-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    5/2500 train=0.05445 lr=1.00e-04
[decompdiff-etth1-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    6/2500 train=0.05347 lr=1.00e-04
[decompdiff-etth1-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    7/2500 train=0.05340 lr=1.00e-04
[decompdiff-etth1-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    8/2500 train=0.05368 lr=1.00e-04
[decompdiff-etth1-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    9

correlational_score,█▇▂▄▁
disc_score,█▇▃▂▁
disc_score_std,█▄█▁▅
epoch,▁▁▁▁▁▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇██
fdds,█▄▁▁▂
lr,████████▇▇▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
pred_mae,█▃▃▁▁
pred_mae_std,▆▁▂█▁
test_acc,█▇▃▂▁
train_loss,█▆▇▅▅▄▄▄▄▃▃▃▃▃▃▃▃▃▃▃▃▃▂▁▂▂▂▂▂▂▂▂▂▂▂▁▂▂▂▂
+1,...


(DecompDiff(
   (decomp): SeriesDecomposition()
   (time_embedder): TimestepEmbedder(
     (sinusoidal): SinusoidalEmbedding()
     (mlp): Sequential(
       (0): Linear(in_features=256, out_features=64, bias=True)
       (1): SiLU()
       (2): Linear(in_features=64, out_features=64, bias=True)
     )
   )
   (trend_input_proj): Linear(in_features=7, out_features=64, bias=True)
   (season_input_proj): Linear(in_features=7, out_features=64, bias=True)
   (res_input_proj): Linear(in_features=7, out_features=64, bias=True)
   (trend_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (season_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (res_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (trend_dit): DiTStack(
     (blocks): ModuleList(
       (0): DiTBlock(
         (norm1): LayerNorm((64,), eps=1e-06, elementwise_affine=False)
         (norm2): LayerNorm((64,), eps=1e-

In [7]:
# ── etth2 · MSE · x0 · loss weight ON ──
run_experiment(dataset="etth2", window_length=WINDOW, num_epochs=EPOCHS,
               num_layers=1, num_fusion_layers=1, hidden_dim=64, batch_size=32, 
               eval_every=EVAL_EVERY)

StockDataset: 17389 windows  (train=17389, test=all [train_ratio=1.0])


[decompdiff-etth2-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] batches=544 channels=7 batch_size=32
[decompdiff-etth2-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] model_dim=64 params=252,807
[decompdiff-etth2-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    1/2500 train=0.12654 lr=1.00e-04
[decompdiff-etth2-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    2/2500 train=0.04230 lr=1.00e-04
[decompdiff-etth2-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    3/2500 train=0.03899 lr=1.00e-04
[decompdiff-etth2-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    4/2500 train=0.03812 lr=1.00e-04
[decompdiff-etth2-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    5/2500 train=0.03656 lr=1.00e-04
[decompdiff-etth2-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    6/2500 train=0.03616 lr=1.00e-04
[decompdiff-etth2-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    7/2500 train=0.03596 lr=1.00e-04
[decompdiff-etth2-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    8/2500 train=0.03507 lr=1.00e-04
[decompdiff-etth2-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    9

correlational_score,▄█▂▁▁
disc_score,█▅▄▁▁
disc_score_std,▄█▃▃▁
epoch,▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇██
fdds,█▁▂▁▂
lr,██████▇▇▇▇▇▇▆▆▆▆▅▅▄▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
pred_mae,▄▅█▁▃
pred_mae_std,▄▂█▃▁
test_acc,█▅▄▁▁
train_loss,█▅▄▄▃▃▃▃▂▂▂▂▂▂▂▁▂▂▂▂▂▂▂▁▂▂▁▂▁▁▁▁▂▂▂▁▁▁▁▁
+1,...


(DecompDiff(
   (decomp): SeriesDecomposition()
   (time_embedder): TimestepEmbedder(
     (sinusoidal): SinusoidalEmbedding()
     (mlp): Sequential(
       (0): Linear(in_features=256, out_features=64, bias=True)
       (1): SiLU()
       (2): Linear(in_features=64, out_features=64, bias=True)
     )
   )
   (trend_input_proj): Linear(in_features=7, out_features=64, bias=True)
   (season_input_proj): Linear(in_features=7, out_features=64, bias=True)
   (res_input_proj): Linear(in_features=7, out_features=64, bias=True)
   (trend_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (season_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (res_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (trend_dit): DiTStack(
     (blocks): ModuleList(
       (0): DiTBlock(
         (norm1): LayerNorm((64,), eps=1e-06, elementwise_affine=False)
         (norm2): LayerNorm((64,), eps=1e-

In [8]:
# ── exchange · MSE · x0 · loss weight ON ──
run_experiment(dataset="exchange", window_length=WINDOW, num_epochs=EPOCHS,
               num_layers=1, num_fusion_layers=1, hidden_dim=64, batch_size=32,
               eval_every=EVAL_EVERY)

StockDataset: 7557 windows  (train=7557, test=all [train_ratio=1.0])


[decompdiff-exchange-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] batches=237 channels=8 batch_size=32
[decompdiff-exchange-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] model_dim=64 params=253,064
[decompdiff-exchange-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    1/2500 train=0.36390 lr=1.00e-04
[decompdiff-exchange-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    2/2500 train=0.05105 lr=1.00e-04
[decompdiff-exchange-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    3/2500 train=0.04406 lr=1.00e-04
[decompdiff-exchange-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    4/2500 train=0.04214 lr=1.00e-04
[decompdiff-exchange-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    5/2500 train=0.04029 lr=1.00e-04
[decompdiff-exchange-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    6/2500 train=0.03950 lr=1.00e-04
[decompdiff-exchange-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    7/2500 train=0.03814 lr=1.00e-04
[decompdiff-exchange-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    8/2500 train=0.03735 lr=1.00e-04
[decompdiff-exchange-L32-H64-NL1-

correlational_score,█▆▃▁▂
disc_score,█▂▇▁▂
disc_score_std,▄▁▆▇█
epoch,▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇██
fdds,█▄▂▃▁
lr,██████████▇▇▇▇▇▆▆▆▆▆▆▆▆▆▅▄▄▄▃▃▃▂▂▂▂▁▁▁▁▁
pred_mae,▅▇▇█▁
pred_mae_std,▇▄█▅▁
test_acc,█▂▇▁▁
train_loss,█▆▃▃▄▂▃▄▄▃▃▃▃▂▃▂▃▂▂▃▂▂▃▃▂▃▃▂▂▃▂▂▁▂▂▂▁▃▂▃
+1,...


(DecompDiff(
   (decomp): SeriesDecomposition()
   (time_embedder): TimestepEmbedder(
     (sinusoidal): SinusoidalEmbedding()
     (mlp): Sequential(
       (0): Linear(in_features=256, out_features=64, bias=True)
       (1): SiLU()
       (2): Linear(in_features=64, out_features=64, bias=True)
     )
   )
   (trend_input_proj): Linear(in_features=8, out_features=64, bias=True)
   (season_input_proj): Linear(in_features=8, out_features=64, bias=True)
   (res_input_proj): Linear(in_features=8, out_features=64, bias=True)
   (trend_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (season_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (res_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (trend_dit): DiTStack(
     (blocks): ModuleList(
       (0): DiTBlock(
         (norm1): LayerNorm((64,), eps=1e-06, elementwise_affine=False)
         (norm2): LayerNorm((64,), eps=1e-

In [9]:
# ── fmri · MSE · x0 · loss weight ON ──
run_experiment(dataset="fmri", window_length=WINDOW, num_epochs=EPOCHS,
               num_layers=1, num_fusion_layers=1, hidden_dim=64, batch_size=32,
               eval_every=EVAL_EVERY)

StockDataset: 9969 windows  (train=9969, test=all [train_ratio=1.0])


[decompdiff-fmri-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] batches=312 channels=50 batch_size=32
[decompdiff-fmri-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] model_dim=64 params=263,858
[decompdiff-fmri-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    1/2500 train=0.21621 lr=1.00e-04
[decompdiff-fmri-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    2/2500 train=0.11604 lr=1.00e-04
[decompdiff-fmri-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    3/2500 train=0.10708 lr=1.00e-04
[decompdiff-fmri-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    4/2500 train=0.10301 lr=1.00e-04
[decompdiff-fmri-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    5/2500 train=0.10087 lr=1.00e-04
[decompdiff-fmri-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    6/2500 train=0.09897 lr=1.00e-04
[decompdiff-fmri-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    7/2500 train=0.09766 lr=1.00e-04
[decompdiff-fmri-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    8/2500 train=0.09744 lr=1.00e-04
[decompdiff-fmri-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    9/2500 trai

correlational_score,█▇▅▃▁
disc_score,██▅▃▁
disc_score_std,▁▁▅▁█
epoch,▁▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▇▇███
fdds,█▇▄▂▁
lr,██████▇▇▇▇▇▇▇▇▇▅▅▅▅▅▄▄▃▃▃▃▃▃▃▃▃▃▂▂▂▁▁▁▁▁
pred_mae,█▄▃▃▁
pred_mae_std,▁▄▆█▂
test_acc,██▅▃▁
train_loss,▇█▇▇▇▆▅▇▇▆▄▅▅▅▅▄▄▄▃▃▃▂▃▂▂▃▃▃▁▂▂▂▁▂▂▃▂▃▂▂
+1,...


(DecompDiff(
   (decomp): SeriesDecomposition()
   (time_embedder): TimestepEmbedder(
     (sinusoidal): SinusoidalEmbedding()
     (mlp): Sequential(
       (0): Linear(in_features=256, out_features=64, bias=True)
       (1): SiLU()
       (2): Linear(in_features=64, out_features=64, bias=True)
     )
   )
   (trend_input_proj): Linear(in_features=50, out_features=64, bias=True)
   (season_input_proj): Linear(in_features=50, out_features=64, bias=True)
   (res_input_proj): Linear(in_features=50, out_features=64, bias=True)
   (trend_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (season_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (res_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (trend_dit): DiTStack(
     (blocks): ModuleList(
       (0): DiTBlock(
         (norm1): LayerNorm((64,), eps=1e-06, elementwise_affine=False)
         (norm2): LayerNorm((64,), eps=

In [ ]:
# ── eeg · MSE · x0 · loss weight ON ──
run_experiment(dataset="eeg", window_length=WINDOW, num_epochs=EPOCHS,
               num_layers=1, num_fusion_layers=1, hidden_dim=64, batch_size=32,
               eval_every=EVAL_EVERY)

StockDataset: 14949 windows  (train=14949, test=all [train_ratio=1.0])


[decompdiff-eeg-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] batches=468 channels=14 batch_size=32
[decompdiff-eeg-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] model_dim=64 params=254,606
[decompdiff-eeg-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    1/2500 train=0.28315 lr=1.00e-04
[decompdiff-eeg-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    2/2500 train=0.00208 lr=1.00e-04
[decompdiff-eeg-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    3/2500 train=0.00121 lr=1.00e-04
[decompdiff-eeg-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    4/2500 train=0.00095 lr=1.00e-04
[decompdiff-eeg-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    5/2500 train=0.00088 lr=1.00e-04
[decompdiff-eeg-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    6/2500 train=0.00077 lr=1.00e-04
[decompdiff-eeg-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    7/2500 train=0.00074 lr=1.00e-04
[decompdiff-eeg-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    8/2500 train=0.00073 lr=1.00e-04
[decompdiff-eeg-L32-H64-NL1-NF1-mse-x0-w1-bs32-E2500] ep    9/2500 train=0.00072 l

In [ ]:
# ── sine · MSE · x0 · loss weight ON ──
run_experiment(dataset="sine", window_length=WINDOW, num_epochs=EPOCHS,
               num_layers=1, num_fusion_layers=1, hidden_dim=64, batch_size=32,
               eval_every=EVAL_EVERY)

StockDataset: 10000 windows  (train=10000, test=all [train_ratio=1.0])


[decompdiff-sine-L32-H64-NL1-NF1-mse-x0-w1-bs32-E1000] batches=313 channels=5 batch_size=32
[decompdiff-sine-L32-H64-NL1-NF1-mse-x0-w1-bs32-E1000] model_dim=64 params=252,293
[decompdiff-sine-L32-H64-NL1-NF1-mse-x0-w1-bs32-E1000] ep    1/1000 train=0.54617 lr=1.00e-04
[decompdiff-sine-L32-H64-NL1-NF1-mse-x0-w1-bs32-E1000] ep    2/1000 train=0.14400 lr=1.00e-04
[decompdiff-sine-L32-H64-NL1-NF1-mse-x0-w1-bs32-E1000] ep    3/1000 train=0.11707 lr=1.00e-04
[decompdiff-sine-L32-H64-NL1-NF1-mse-x0-w1-bs32-E1000] ep    4/1000 train=0.11039 lr=1.00e-04
[decompdiff-sine-L32-H64-NL1-NF1-mse-x0-w1-bs32-E1000] ep    5/1000 train=0.10634 lr=1.00e-04
[decompdiff-sine-L32-H64-NL1-NF1-mse-x0-w1-bs32-E1000] ep    6/1000 train=0.10235 lr=1.00e-04
[decompdiff-sine-L32-H64-NL1-NF1-mse-x0-w1-bs32-E1000] ep    7/1000 train=0.10069 lr=1.00e-04
[decompdiff-sine-L32-H64-NL1-NF1-mse-x0-w1-bs32-E1000] ep    8/1000 train=0.09904 lr=1.00e-04
[decompdiff-sine-L32-H64-NL1-NF1-mse-x0-w1-bs32-E1000] ep    9/1000 train

correlational_score,█▁
disc_score,█▁
disc_score_std,▁█
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▄▄▄▄▅▅▅▅▅▅▅▇▇▇▇▇▇▇▇██
fdds,█▁
lr,████████▇▇▇▇▇▇▇▆▆▆▆▅▅▅▄▄▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
pred_mae,█▁
pred_mae_std,█▁
test_acc,█▁
train_loss,█▄▄▃▄▄▂▃▃▃▂▃▂▂▃▁▂▂▂▁▂▂▂▂▃▂▂▂▂▁▂▂▂▂▁▂▂▂▂▁
+1,...


(DecompDiff(
   (decomp): SeriesDecomposition()
   (time_embedder): TimestepEmbedder(
     (sinusoidal): SinusoidalEmbedding()
     (mlp): Sequential(
       (0): Linear(in_features=256, out_features=64, bias=True)
       (1): SiLU()
       (2): Linear(in_features=64, out_features=64, bias=True)
     )
   )
   (trend_input_proj): Linear(in_features=5, out_features=64, bias=True)
   (season_input_proj): Linear(in_features=5, out_features=64, bias=True)
   (res_input_proj): Linear(in_features=5, out_features=64, bias=True)
   (trend_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (season_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (res_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (trend_dit): DiTStack(
     (blocks): ModuleList(
       (0): DiTBlock(
         (norm1): LayerNorm((64,), eps=1e-06, elementwise_affine=False)
         (norm2): LayerNorm((64,), eps=1e-

## Part 2 — stock batch-size sweep (MSE · x0 · loss weight ON)

In [ ]:
# ── stock · batch size 24 · MSE · x0 · loss weight ON ──
run_experiment(dataset="stock", window_length=WINDOW, num_epochs=EPOCHS,
               num_layers=1, num_fusion_layers=1, hidden_dim=64,
               batch_size=24, eval_every=EVAL_EVERY)

StockDataset: 3654 windows  (train=3654, test=all [train_ratio=1.0])


[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs24-E1000] batches=153 channels=6 batch_size=24
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs24-E1000] model_dim=64 params=252,550
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs24-E1000] ep    1/1000 train=0.58037 lr=1.00e-04
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs24-E1000] ep    2/1000 train=0.05194 lr=1.00e-04
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs24-E1000] ep    3/1000 train=0.03305 lr=1.00e-04
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs24-E1000] ep    4/1000 train=0.03024 lr=1.00e-04
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs24-E1000] ep    5/1000 train=0.02791 lr=1.00e-04
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs24-E1000] ep    6/1000 train=0.02568 lr=1.00e-04
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs24-E1000] ep    7/1000 train=0.02668 lr=1.00e-04
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs24-E1000] ep    8/1000 train=0.02515 lr=1.00e-04
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs24-E1000] ep    9

correlational_score,▁█
disc_score,█▁
disc_score_std,▁█
epoch,▁▁▁▁▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇██
fdds,▁█
lr,██████▇▇▇▇▆▆▆▆▆▅▅▅▄▄▄▄▃▃▃▃▃▃▂▂▁▁▁▁▁▁▁▁▁▁
pred_mae,█▁
pred_mae_std,█▁
test_acc,█▁
train_loss,█▇▄▄▃▄▅▄▃▃▂▄▂▅▂▃▂▄▂▁▄▃▄▃▄▃▂▂▂▃▃▂▂▁▁▂▃▂▂▄
+1,...


(DecompDiff(
   (decomp): SeriesDecomposition()
   (time_embedder): TimestepEmbedder(
     (sinusoidal): SinusoidalEmbedding()
     (mlp): Sequential(
       (0): Linear(in_features=256, out_features=64, bias=True)
       (1): SiLU()
       (2): Linear(in_features=64, out_features=64, bias=True)
     )
   )
   (trend_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (season_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (res_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (trend_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (season_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (res_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (trend_dit): DiTStack(
     (blocks): ModuleList(
       (0): DiTBlock(
         (norm1): LayerNorm((64,), eps=1e-06, elementwise_affine=False)
         (norm2): LayerNorm((64,), eps=1e-

In [ ]:
# ── stock · batch size 64 · MSE · x0 · loss weight ON ──
run_experiment(dataset="stock", window_length=WINDOW, num_epochs=EPOCHS,
               num_layers=1, num_fusion_layers=1, hidden_dim=64,
               batch_size=64, eval_every=EVAL_EVERY)

StockDataset: 3654 windows  (train=3654, test=all [train_ratio=1.0])


[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs64-E1000] batches=58 channels=6 batch_size=64
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs64-E1000] model_dim=64 params=252,550
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs64-E1000] ep    1/1000 train=0.71833 lr=5.80e-05
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs64-E1000] ep    2/1000 train=0.30553 lr=1.00e-04
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs64-E1000] ep    3/1000 train=0.07914 lr=1.00e-04
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs64-E1000] ep    4/1000 train=0.04600 lr=1.00e-04
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs64-E1000] ep    5/1000 train=0.03641 lr=1.00e-04
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs64-E1000] ep    6/1000 train=0.03296 lr=1.00e-04
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs64-E1000] ep    7/1000 train=0.03132 lr=1.00e-04
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs64-E1000] ep    8/1000 train=0.02904 lr=1.00e-04
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs64-E1000] ep    9/

correlational_score,█▁
disc_score,█▁
disc_score_std,█▁
epoch,▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇█
fdds,▁█
lr,████████▇▇▇▇▇▇▇▇▆▆▆▆▅▅▅▅▄▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁
pred_mae,█▁
pred_mae_std,█▁
test_acc,█▁
train_loss,█▅▅▄▄▄▂▂▂▄▃▃▃▃▃▁▂▃▁▃▃▃▂▄▂▂▂▂▂▂▂▂▁▂▂▂▃▂▂▃
+1,...


(DecompDiff(
   (decomp): SeriesDecomposition()
   (time_embedder): TimestepEmbedder(
     (sinusoidal): SinusoidalEmbedding()
     (mlp): Sequential(
       (0): Linear(in_features=256, out_features=64, bias=True)
       (1): SiLU()
       (2): Linear(in_features=64, out_features=64, bias=True)
     )
   )
   (trend_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (season_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (res_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (trend_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (season_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (res_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (trend_dit): DiTStack(
     (blocks): ModuleList(
       (0): DiTBlock(
         (norm1): LayerNorm((64,), eps=1e-06, elementwise_affine=False)
         (norm2): LayerNorm((64,), eps=1e-

In [ ]:
# ── stock · batch size 128 · MSE · x0 · loss weight ON ──
run_experiment(dataset="stock", window_length=WINDOW, num_epochs=EPOCHS,
               num_layers=1, num_fusion_layers=1, hidden_dim=64,
               batch_size=128, eval_every=EVAL_EVERY)

StockDataset: 3654 windows  (train=3654, test=all [train_ratio=1.0])


[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs128-E1000] batches=29 channels=6 batch_size=128
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs128-E1000] model_dim=64 params=252,550
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs128-E1000] ep    1/1000 train=0.95717 lr=2.91e-05
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs128-E1000] ep    2/1000 train=0.86542 lr=5.80e-05
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs128-E1000] ep    3/1000 train=0.56609 lr=8.70e-05
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs128-E1000] ep    4/1000 train=0.21905 lr=1.00e-04
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs128-E1000] ep    5/1000 train=0.08408 lr=1.00e-04
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs128-E1000] ep    6/1000 train=0.05765 lr=1.00e-04
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs128-E1000] ep    7/1000 train=0.04545 lr=1.00e-04
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs128-E1000] ep    8/1000 train=0.03853 lr=1.00e-04
[decompdiff-stock-L32-H64-NL1-NF1-mse-x0-w1-bs128-E10

correlational_score,█▁
disc_score,█▁
disc_score_std,█▁
epoch,▁▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇████
fdds,▁█
lr,███████▇▇▇▆▆▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▁▁▁▁▁
pred_mae,▁█
pred_mae_std,▁█
test_acc,█▁
train_loss,▆█▇▇▅▃▁▇▄▆▄▅▄▂▂▃▁▄▅▃▁▂▂▆▃▃▂▃▂▄▅▄▁▆▃▄▂▁▄▂
+1,...


(DecompDiff(
   (decomp): SeriesDecomposition()
   (time_embedder): TimestepEmbedder(
     (sinusoidal): SinusoidalEmbedding()
     (mlp): Sequential(
       (0): Linear(in_features=256, out_features=64, bias=True)
       (1): SiLU()
       (2): Linear(in_features=64, out_features=64, bias=True)
     )
   )
   (trend_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (season_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (res_input_proj): Linear(in_features=6, out_features=64, bias=True)
   (trend_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (season_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (res_pe): LearnablePositionalEncoding(
     (dropout): Dropout(p=0.0, inplace=False)
   )
   (trend_dit): DiTStack(
     (blocks): ModuleList(
       (0): DiTBlock(
         (norm1): LayerNorm((64,), eps=1e-06, elementwise_affine=False)
         (norm2): LayerNorm((64,), eps=1e-